In [126]:
import numpy as np

from load_mtx import load_mtx

A, b = load_mtx("tasks/1_3.txt")
n = len(b)

A, b

(array([[ 29.,   8.,   9.,  -9.],
        [ -7., -25.,   0.,   9.],
        [  1.,   6.,  16.,  -2.],
        [ -7.,   4.,  -2.,  17.]]),
 array([ 197., -226.,  -95.,  -58.]))

Введём функции нормы

In [127]:
def matrix_norm(matrix):
    matrix = np.asarray(matrix)
    return matrix.sum(axis=1).max()

def vector_norm(vector):
    vector = np.asarray(vector)
    return np.abs(vector).max()

Точное решение СЛАУ:

In [128]:
from IPython.display import display, Math

actual = np.linalg.solve(A, b)
for i in range(n):
    display(Math(fr"x_{i + 1} = {actual[i]:.3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Метод простых итераций

\begin{aligned}
\text{Найдём }\alpha\text{ и }\beta
\end{aligned}

In [129]:
import pandas as pd

alpha = A.copy()
beta = b.copy()

for i in range(n):
    beta[i] /= alpha[i, i]
    alpha[i] /= -alpha[i, i]
    alpha[i, i] = 0

display(pd.DataFrame(alpha), pd.DataFrame(beta))

,0,1,2,3
0,0.000000,-0.275862,-0.310345,0.310345
1,-0.280000,0.000000,0.000000,0.360000
2,-0.062500,-0.375000,0.000000,0.125000
3,0.411765,-0.235294,0.117647,0.000000


,0
0,6.793103
1,9.040000
2,-5.937500
3,-3.411765



\begin{aligned}
\text{Зададим точность }\varepsilon &= 0.001
\end{aligned}

In [130]:
eps = 1e-3

In [131]:
actual = np.linalg.solve(A, b)
for i in range(n):
    display(Math(fr"x_{i+1} = {actual[i]:.3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [132]:
norm_alpha = matrix_norm(A)
norm_beta = vector_norm(b)

iteration = 1
prev_x = beta.copy()

In [133]:
while True:
    x_iteration = beta + alpha @ prev_x
    eps_i = vector_norm(x_iteration - prev_x)
    if norm_alpha < 1:
        eps_i *= norm_alpha / (1 - norm_alpha)

    if eps_i <= eps:
        break

    prev_x = x_iteration
    iteration += 1

In [134]:
for i in range(n):
    display(Math(fr"x_{i+1} = {x_iteration[i]:.3f}"))

print(f"{iteration} итераций")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

17 итераций


# Метод Зейделя

In [135]:
C, B = np.triu(alpha), np.tril(alpha, k=-1)
# верхняя, с главной диагональю и нижняя, без главной диагонали
norm_c = matrix_norm(C)

iteration = 1
prev_x = beta.copy()

In [136]:
while True:
    x_seidel = prev_x.copy()

    for i in range(n):
        x_seidel[i] = beta[i] + alpha[i] @ x_seidel

    eps_i = vector_norm(x_seidel - prev_x)
    if norm_c < 1:
        eps_i *= norm_c / (1 - norm_c)

    if eps_i <= eps:
        break

    prev_x = x_seidel
    iteration += 1

In [137]:
for i in range(n):
    display(Math(fr"x_{i+1} = {x_seidel[i]:.3f}"))

print(f"{iteration} итераций")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

7 итераций


Проверка решения

In [138]:
np.allclose(x_seidel, x_iteration)

False

In [139]:
np.allclose(x_seidel, actual)

False